<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-3-rag-pipelines/Module_3_Session_2_Vector_Store_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 - Session 2: Real Vector Store with FAISS
Goal: Store Swiggy FAQ embeddings in a real vector database and retrieve instantly.

In [1]:
# Install all libraries we need for this session
# sentence-transformers — same as Session 1, converts text to vectors
# faiss-cpu — Facebook's vector database, stores and searches embeddings fast
# groq — to generate the final answer using LLM
!pip install -q sentence-transformers faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.7 MB/s eta 0:00:00


# Module 3 - Session 2: Real Vector Store with FAISS
Goal: Store Swiggy FAQ embeddings in a real vector database and retrieve instantly.

In [2]:
# Install all libraries we need for this session
# sentence-transformers — converts text to vectors (same as Session 1)
# faiss-cpu — Facebook's vector database, stores and searches embeddings fast
# groq — to generate the final LLM answer
!pip install -q sentence-transformers faiss-cpu groq

## Step 2 - Load the Embedding Model
Same model as Session 1 — all-MiniLM-L6-v2.
Converts any sentence into a vector of 384 numbers.

In [3]:
# Import SentenceTransformer to convert text to vectors
from sentence_transformers import SentenceTransformer

# Load the same model we used in Session 1
# We are reusing this because consistency matters —
# embeddings stored and queries MUST use the same model
# If you store with model A and search with model B, results will be garbage
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


## Step 3 - Define Swiggy FAQ Knowledge Base
These are the documents stored in our vector database.
In a real system this would be thousands of help articles loaded from a file.

In [4]:
# Our Swiggy FAQ knowledge base
# In Session 1 we had 6 questions
# Today we have 15 — still small but enough to feel the difference
# In a real enterprise system this would be 50,000+ articles loaded from S3
swiggy_faqs = [
    "Where is my order?",
    "How do I cancel my order?",
    "My food arrived cold, what should I do?",
    "How do I apply a coupon code?",
    "My payment failed but money was deducted.",
    "How do I change my delivery address?",
    "How do I contact my delivery partner?",
    "My order was delivered to the wrong address.",
    "How do I get a refund for my cancelled order?",
    "The restaurant cancelled my order, what happens now?",
    "How do I rate my delivery experience?",
    "My Swiggy One membership is not working.",
    "How do I update my phone number on Swiggy?",
    "Can I schedule an order for later?",
    "How do I report a missing item in my order?"
]

print(f"Knowledge base has {len(swiggy_faqs)} FAQ questions")

Knowledge base has 15 FAQ questions


## Step 4 - Convert FAQ Questions into Embeddings
We convert all 15 FAQs into vectors.
This happens ONCE at setup time — not every time a customer asks a question.

In [5]:
# Convert all 15 FAQ questions into embeddings
# .encode() returns a numpy array of shape (15, 384)
# 15 questions, each represented as 384 numbers
faq_embeddings = model.encode(swiggy_faqs)

print(f"Shape of FAQ embeddings: {faq_embeddings.shape}")
print(f"Each FAQ is now a vector of {faq_embeddings.shape[1]} numbers")

Shape of FAQ embeddings: (15, 384)
Each FAQ is now a vector of 384 numbers


## Step 5 - Build the FAISS Vector Database
We store all 15 FAQ embeddings in FAISS.
FAISS can search millions of vectors in milliseconds.
This is the "library shelf" where all our knowledge lives.

In [6]:
# Import FAISS — Facebook AI Similarity Search
import faiss
import numpy as np

# FAISS requires embeddings in float32 format
# Our embeddings are already numpy arrays but we ensure correct type
faq_embeddings_float = np.array(faq_embeddings).astype("float32")

# Create a FAISS index
# IndexFlatL2 means — store vectors and search using L2 distance (straight line distance)
# 384 is the size of each vector — must match our embedding model output
index = faiss.IndexFlatL2(384)

# Add all 15 FAQ embeddings into the FAISS index
# This is like putting all books onto the library shelf
index.add(faq_embeddings_float)

print(f"FAISS index built successfully!")
print(f"Total vectors stored in FAISS: {index.ntotal}")

FAISS index built successfully!
Total vectors stored in FAISS: 15


## Step 6 - Search FAISS with Customer Query
This is the ONLINE step — happens every time a customer asks a question.
We convert the query to a vector and FAISS instantly finds the closest FAQ.

In [7]:
# This is what the customer typed
customer_query = "my order hasn't arrived yet and it's been 45 minutes"

# Convert customer query to embedding — same model, same 384 numbers
# This is the ONLINE step — happens in real time, per query
query_embedding = model.encode([customer_query])

# Convert to float32 — FAISS requires this format
query_embedding_float = np.array(query_embedding).astype("float32")

# Search FAISS for the top 3 closest FAQs
# k=3 means — give me the 3 most relevant results
# This is called "top-k retrieval" — a key RAG concept
distances, indices = index.search(query_embedding_float, k=3)

# Print the top 3 results
print(f"Customer query: '{customer_query}'\n")
print("Top 3 most relevant FAQs:")
print("-" * 50)

for rank, (distance, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank {rank+1} — Score: {distance:.4f} → {swiggy_faqs[idx]}")

Customer query: 'my order hasn't arrived yet and it's been 45 minutes'

Top 3 most relevant FAQs:
--------------------------------------------------
Rank 1 — Score: 0.8414 → My order was delivered to the wrong address.
Rank 2 — Score: 0.8535 → Where is my order?
Rank 3 — Score: 1.1234 → How do I report a missing item in my order?


The difference is tiny — 0.8414 vs 0.8535 — basically a coin flip. This happens because IndexFlatL2 uses straight line distance which is slightly less accurate for sentence meaning than cosine similarity.
This is exactly the kind of thing FAANG interviewers want you to notice — not just "did it work" but "why did it rank this way and how do I fix it."

## Step 7 - Rebuild FAISS with Cosine Similarity
IndexFlatL2 uses straight line distance.
IndexFlatIP uses cosine similarity — more accurate for sentence meaning.
We normalize vectors first, then use IndexFlatIP.

In [8]:
# Normalize embeddings before storing
# Normalization means we scale every vector to length 1
# When vectors are normalized, L2 distance and cosine similarity give same result
# faiss.normalize_L2 does this in one line
faiss.normalize_L2(faq_embeddings_float)

# IndexFlatIP = Inner Product = cosine similarity on normalized vectors
# This is more accurate for sentence meaning than IndexFlatL2
index_cosine = faiss.IndexFlatIP(384)

# Add normalized FAQ embeddings
index_cosine.add(faq_embeddings_float)

print(f"FAISS cosine index built successfully!")
print(f"Total vectors stored: {index_cosine.ntotal}")

FAISS cosine index built successfully!
Total vectors stored: 15


## Step 8 - Search Again with Cosine Similarity Index
Now higher score = more similar (opposite of L2).
We expect "Where is my order?" to rank first this time.

In [9]:
# Normalize the query embedding too — must match how we stored the FAQs
# This is a common mistake — people normalize the index but forget the query
faiss.normalize_L2(query_embedding_float)

# Search the cosine index for top 3
# Now higher score = more similar (opposite of L2 where lower was better)
distances, indices = index_cosine.search(query_embedding_float, k=3)

# Print results
print(f"Customer query: '{customer_query}'\n")
print("Top 3 most relevant FAQs (cosine similarity):")
print("-" * 50)

for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank {rank+1} — Score: {score:.4f} → {swiggy_faqs[idx]}")

Customer query: 'my order hasn't arrived yet and it's been 45 minutes'

Top 3 most relevant FAQs (cosine similarity):
--------------------------------------------------
Rank 1 — Score: 0.5793 → My order was delivered to the wrong address.
Rank 2 — Score: 0.5733 → Where is my order?
Rank 3 — Score: 0.4383 → How do I report a missing item in my order?


## Step 9 - Send Retrieved FAQs to LLM
We pass the top 2 retrieved FAQs as context.
The LLM reads both and generates a grounded answer.
In production we always send top-k results, not just top 1.

In [10]:
import os
from groq import Groq
from google.colab import userdata

# Load Groq API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

# Collect top 2 retrieved FAQs as context
# In production top 3 or top 5 is common — more context = better answer
retrieved_faqs = [swiggy_faqs[idx] for idx in indices[0][:2]]

# Build context string from retrieved FAQs
context = "\n".join([f"- {faq}" for faq in retrieved_faqs])

# Build the prompt
prompt = f"""You are a Swiggy customer support agent.

Use the following retrieved FAQ context to answer the customer question.
Only use the context provided — do not make up information.

Retrieved Context:
{context}

Customer Question: {customer_query}

Reply in a friendly, helpful tone in 2-3 sentences."""

# Send to LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(f"Customer asked: '{customer_query}'\n")
print(f"Retrieved context:\n{context}\n")
print(f"Swiggy Support Reply:")
print(response.choices[0].message.content)

Customer asked: 'my order hasn't arrived yet and it's been 45 minutes'

Retrieved context:
- My order was delivered to the wrong address.
- Where is my order?

Swiggy Support Reply:
We're here to help you track down your order. Our delivery partners aim to reach you within 45 minutes to an hour, but if it's been longer than that, let's check on the status together. Can you please provide me with the order number so I can look into it for you?


## What We Built
- Loaded 15 Swiggy FAQ questions as our knowledge base
- Converted all 15 into embeddings using sentence-transformers (OFFLINE step)
- Stored embeddings in a FAISS vector database with cosine similarity
- Took a customer query, converted it to embedding (ONLINE step)
- Retrieved top 2 most relevant FAQs from FAISS instantly
- Sent retrieved context + query to Groq LLM for a grounded answer

## Key Concepts Learned
- OFFLINE vs ONLINE processing — embed documents once, query in real time
- IndexFlatL2 vs IndexFlatIP — L2 distance vs cosine similarity in FAISS
- Normalizing vectors — must normalize both stored vectors AND query
- Top-k retrieval — send multiple results to LLM, let it reason across them
- RAG quality depends 60% on knowledge base quality, 40% on algorithm

## AWS Equivalent
| What we did here         | AWS service                          |
|--------------------------|--------------------------------------|
| FAISS index              | Amazon OpenSearch Serverless         |
| sentence-transformers    | Amazon Titan Embeddings              |
| Storing embeddings       | Amazon Bedrock Knowledge Bases       |
| Groq LLM                 | Amazon Bedrock (Claude/Llama)        |

## What is Next
Session 3 — Full RAG Pipeline
Wire everything together with a real PDF document.
Customer asks a question → PDF gets searched → LLM answers from the document.